# Validating `null_basis_realization` (beyond the TIB balance check)

The TIB balance identity `A Aᵀ + B Bᵀ = I` is **necessary but not sufficient** — it holds for *any* unit null vectors, so it can't tell whether the Blaschke–Potapov deflation (the part that depends on the null vectors) was done correctly. A wrong null-vector / Blaschke step still passes balance while realizing the **wrong transfer function** (this is how the historical line-185 argument-swap bug slipped through).

This notebook adds the checks that *do* test the transfer function.

**What we use (and its limits).**
- The code's Blaschke factor `b_w(z)=(z−w)/(1−w̄z)` is the *inner* (pole-outside) convention; `null_basis_realization` builds the *stable* realization (poles `λ` inside). The matching stable elementary lossless factor is the reciprocal `b̃_w(z)=(1−wz)/(z−w)`.
- Verified below: for one mode, `null_basis_realization` returns `A=[λ]`, `B=√(1−λ²)·v*`, and the algorithm's `Jₖ = I−(1+λ)vv*` is exactly the **feedthrough `D`** of the elementary factor `G_{λ,v}`. So **`null_basis_realization` realizes the cascade `∏ₖ G_{λₖ,yₖ}`** of stable elementary lossless factors (with the Schur-orthonormalized `yₖ`).
- The strongest check here is an **independent re-implementation** (our own deflation with the correct `βλ,z` convention + a pointwise factor product) compared at the transfer-function level. It catches **coding / convention bugs** in `null_basis_realization` (e.g. the line-185 arg swap). It shares `blaschke_potapov_factor` itself; a fully convention-independent test would need the tangential-interpolation condition from `Olivi2010` / `HanzonOliviPeeters2010`.

In [1]:
%matplotlib inline
import sys, os
try:
    import ga  # noqa
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import scipy.linalg as la
from ga.filters.tib import null_basis_realization, blaschke_potapov_factor, poles_chebyshev_roots

rng = np.random.default_rng(1)

# A MIMO test system (q>=2 is essential: SISO can't exercise the null-vector machinery)
q, n = 2, 3
poles = poles_chebyshev_roots(n, 0.0, 0.85)            # real, distinct, inside the disk
v = rng.standard_normal((q, n)); v /= np.linalg.norm(v, axis=0)   # unit null vectors
sys = null_basis_realization(poles, v)
A, B = sys.A(dense=True).real, sys.B(dense=True).real
print(f"q={q} inputs, n={n} modes, poles={np.round(poles,3)}")
print(f"A {A.shape}, B {B.shape}")

q=2 inputs, n=3 modes, poles=[0.057 0.425 0.793]
A (3, 3), B (3, 2)


## 1. Necessary checks — poles + balance (neither tests the transfer function)

In [2]:
bal = np.max(np.abs(A @ A.T + B @ B.T - np.eye(n)))
dpole = np.max(np.abs(np.sort(np.real(la.eigvals(A))) - np.sort(poles)))
print(f"TIB balance |A A* + B B* - I| = {bal:.2e}")
print(f"pole recovery  max|eig(A) - poles| = {dpole:.2e}")
print()
print("(both necessary; both pass for ANY unit null vectors -> they do NOT validate the")
print(" transfer function / null-vector handling)")

TIB balance |A A* + B B* - I| = 2.22e-16
pole recovery  max|eig(A) - poles| = 0.00e+00

(both necessary; both pass for ANY unit null vectors -> they do NOT validate the
 transfer function / null-vector handling)


## 2. Single-mode exact check

For one mode, `null_basis_realization` must equal the elementary stable lossless factor `G_{λ,v}`: `A=[λ]`, `B=√(1−λ²) v*`, and `J₁ = I−(1+λ)vv*` (the factor's feedthrough `D`).

In [3]:
lam = 0.6
u = rng.standard_normal(q); u /= np.linalg.norm(u)
s1 = null_basis_realization([lam], u.reshape(q, 1))
A1, B1 = s1.A(dense=True).real, s1.B(dense=True).real
print("A1            =", A1.ravel(), "   expected [λ] =", [lam])
print("B1            =", B1.ravel(), "   expected √(1-λ²)·v* =", np.round(np.sqrt(1-lam**2)*u, 4))
print("max |B1 - √(1-λ²)v*| =", np.max(np.abs(B1.ravel() - np.sqrt(1-lam**2)*u)))

A1            = [0.6]    expected [λ] = [0.6]
B1            = [-0.54291692  0.58757231]    expected √(1-λ²)·v* = [-0.5429  0.5876]
max |B1 - √(1-λ²)v*| = 1.1102230246251565e-16


## 3. Lossless completion + all-pass

`(A,B)` are the controllable part of a lossless (inner) system. Complete to a unitary `[[A,B],[C,D]]` (orthonormal complement of the rows of `[A B]`); then `Θ(z)=D+C(zI−A)⁻¹B` and `|Θ(z)|` must be **unitary on the unit circle** (all singular values 1).

In [4]:
AB = np.hstack([A, B])
CD = la.null_space(AB).conj().T            # q x (n+q): orthonormal complement of rows of [A B]
C, D = CD[:, :n], CD[:, n:]
Ufull = np.block([[A, B], [C, D]])
print("completion unitary? |U Uᵀ - I| =", np.max(np.abs(Ufull @ Ufull.conj().T - np.eye(n+q))))

def Theta_null(z):
    return D + C @ np.linalg.inv(z*np.eye(n) - A) @ B

zs = np.exp(1j*np.array([0.3, 0.9, 1.7, 2.4, -0.6, 2.0]))
svmin = min(np.linalg.svd(Theta_null(z), compute_uv=False).min() for z in zs)
svmax = max(np.linalg.svd(Theta_null(z), compute_uv=False).max() for z in zs)
print(f"all-pass on |z|=1: singular values in [{svmin:.6f}, {svmax:.6f}]  (should be ~1)")

completion unitary? |U Uᵀ - I| = 5.333691017409846e-16
all-pass on |z|=1: singular values in [1.000000, 1.000000]  (should be ~1)


## 4. Determinant = Blaschke product (independent pole check)

`det G_{w,u} = b̃_w(z)`, so `det Θ(z) = ∏ₖ b̃_{λₖ}(z)` up to a unimodular constant — an independent confirmation of the poles via the transfer-function determinant (a different route from `eig(A)`).

In [5]:
def tb(w, z):    # stable scalar all-pass b̃_w(z), real w
    return (1 - w*z) / (z - w)

ratios = []
for z in zs:
    prod = np.prod([tb(w, z) for w in poles])
    ratios.append(np.linalg.det(Theta_null(z)) / prod)
ratios = np.array(ratios)
print("det Θ(z) / ∏ b̃_λ(z):  |·| =", np.round(np.abs(ratios), 6))
print("  constant (same complex number)?  spread =", f"{np.max(np.abs(ratios - ratios[0])):.2e}")

det Θ(z) / ∏ b̃_λ(z):  |·| = [1. 1. 1. 1. 1. 1.]
  constant (same complex number)?  spread = 1.18e-15


## 5. Independent re-implementation — the real correctness test

Recompute the Schur-orthonormalized `yₖ` ourselves (own deflation, **correct** `βλ,z` convention), build `Θ_casc(z) = ∏ₖ G_{λₖ,yₖ}(z)` by pointwise matrix product, and compare to `Θ_null`. Two lossless realizations of the same function differ only by a **constant** left-unitary (the completion gauge), so `M(z) = Θ_null(z) Θ_casc(z)⁻¹` must be **constant in z**.

In [6]:
def my_deflation(poles, v):                 # our own y_k (correct beta convention)
    q, n = v.shape
    y = np.zeros((q, n))
    y[:, 0] = v[:, 0] / np.linalg.norm(v[:, 0])
    for k in range(1, n):
        M = np.eye(q, dtype=complex)
        for i in range(k):
            beta = blaschke_potapov_factor(poles[i], np.conj(poles[k]), y[:, i])  # pole=λ_i, eval=λ_k*
            M = M @ beta.conj().T
        yk = M @ v[:, k].reshape(-1, 1)
        y[:, k] = np.real(yk / np.linalg.norm(yk)).ravel()
    return y

def Gz(w, uu, z):
    uu = uu.reshape(-1, 1)
    return np.eye(uu.shape[0]) + (tb(w, z) - 1) * (uu @ uu.conj().T)

def Theta_casc(z, y, order):
    M = np.eye(q, dtype=complex)
    for k in order:
        M = M @ Gz(poles[k], y[:, k], z)
    return M

y = my_deflation(poles, v)
order = list(range(n-1, -1, -1))            # reverse order (determined empirically)
Ms = [Theta_null(z) @ np.linalg.inv(Theta_casc(z, y, order)) for z in zs]
spread = max(np.max(np.abs(Ms[i] - Ms[0])) for i in range(len(Ms)))
print(f"M(z) = Θ_null(z) · Θ_casc(z)⁻¹   spread over z = {spread:.2e}")
print("=> CONSTANT  ⇒  null_basis_realization realizes the correct transfer function ✓"
      if spread < 1e-9 else "=> NOT constant ⇒ mismatch")

M(z) = Θ_null(z) · Θ_casc(z)⁻¹   spread over z = 1.25e-15
=> CONSTANT  ⇒  null_basis_realization realizes the correct transfer function ✓


### Does the check have teeth? Inject the line-185 bug

Repeat with the `βλ,z` arguments **swapped** in *our* deflation (the historical bug). If the check is meaningful, the cascade should now disagree with `null_basis_realization` (which uses the correct convention) — `M(z)` no longer constant.

In [7]:
def my_deflation_buggy(poles, v):           # WRONG: blaschke args swapped (line-185 bug)
    q, n = v.shape
    y = np.zeros((q, n)); y[:, 0] = v[:, 0] / np.linalg.norm(v[:, 0])
    for k in range(1, n):
        M = np.eye(q, dtype=complex)
        for i in range(k):
            beta = blaschke_potapov_factor(np.conj(poles[k]), poles[i], y[:, i])  # <-- swapped
            M = M @ beta.conj().T
        yk = M @ v[:, k].reshape(-1, 1)
        y[:, k] = np.real(yk / np.linalg.norm(yk)).ravel()
    return y

y_bug = my_deflation_buggy(poles, v)
Ms_bug = [Theta_null(z) @ np.linalg.inv(Theta_casc(z, y_bug, order)) for z in zs]
spread_bug = max(np.max(np.abs(Ms_bug[i] - Ms_bug[0])) for i in range(len(Ms_bug)))
print(f"with swapped β args:  M(z) spread = {spread_bug:.2e}")
print("=> NOT constant ⇒ the check DETECTS the arg-swap bug ✓" if spread_bug > 1e-6
      else "=> still constant (check would have missed it)")

with swapped β args:  M(z) spread = 1.01e+00
=> NOT constant ⇒ the check DETECTS the arg-swap bug ✓


## Conclusions

- **Balance + pole recovery** pass for any unit null vectors → necessary, not sufficient (they don't test the transfer function).
- **Single-mode** matches the elementary lossless factor exactly; the realization is **all-pass** (lossless) on the unit circle; `det Θ = ∏ b̃_λ` confirms the poles independently.
- **Independent re-implementation** (own deflation, correct `βλ,z` convention, pointwise factor product) reproduces `null_basis_realization`'s transfer function up to the completion gauge (`M(z)` constant) — so the realization is **correct**, and the **arg-swap bug is detected** when injected.
- *Limit:* this shares `blaschke_potapov_factor`; a fully convention-independent check would verify the tangential-interpolation condition from `Olivi2010` / `HanzonOliviPeeters2010`. The realization is **MIMO** (`q≥2`) on purpose — SISO can't exercise the null-vector directions.